In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import sys
import numpy as np
import networkx as nx

In [2]:
sys.path.append('../src/nlr_ghe_screen/clustering/')
from clustering import (
    seed_cluster, 
    construct_euclidian_dist_matrix,
    find_seed_centroids,
    find_nearby_buildings
)

sys.path.append('../src/nlr_ghe_screen/api/')
from building_footprint_api import (
    get_overture_footprints,
    get_ornl_footprints
)
from osmnx_api import (
    get_road_network,
)

In [3]:
TEST_BBOX = (47.58, -122.36, 47.61, -122.31) # xmin, ymin, xmax, ymax
south, west, north, east = TEST_BBOX

In [10]:
ghe_locations = gpd.read_file("../src/exports/ghe_locations.geojson")
geom_box = box(west, south, east, north)
cropped_ghes = gpd.clip(ghe_locations, geom_box)

In [ ]:
# overture_buildings = get_overture_footprints(bbox=TEST_BBOX)
ornl_buildings = get_ornl_footprints(bbox=TEST_BBOX)

In [ ]:
cropped_ghes = cropped_ghes.to_crs(cropped_ghes.estimate_utm_crs())
ornl_buildings = ornl_buildings.to_crs(cropped_ghes.crs)

dist_matrix = construct_euclidian_dist_matrix(
    buildings_df=cropped_ghes,
    max_distance=250
)

labels = seed_cluster(
    dist_matrix=dist_matrix,
    max_distance=250
)
cropped_ghes['cluster'] = labels

In [ ]:
centroids = find_seed_centroids(
    ghe_locations=cropped_ghes,
    cluster_col='cluster'
)

In [ ]:
matches = find_nearby_buildings(
    seed_cluster_centroids=centroids,
    building_data=ornl_buildings,
    max_distance=250,
)

In [ ]:
# c1 = cropped_ghes['seed_cluster'].value_counts().rename("n1")
# c2 = matches['cluster'].value_counts().rename("n2")

# counts = pd.concat([c1, c2], axis=1).fillna(0).astype(int)
# counts["min"] = counts[["n1", "n2"]].min(axis=1)

# ranked = counts.sort_values("min", ascending=False)      
# ordered_labels = ranked.index.tolist()
# ordered_labels[:10]

In [ ]:
# idx = 5
# m = centroids[centroids['seed_cluster'] == idx].explore(color='red')
# cropped_ghes[cropped_ghes['seed_cluster'] == idx].explore(m=m, color='green')
# matches[matches['cluster'] == idx].explore(m=m, color='blue')